# EPIC Clarity Drug Exposure Hydration

This notebook hydrates the OMOP DRUG_EXPOSURE table from EPIC Clarity medication order data.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_MED` - Medication orders
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_MED_SIG` - Medication instructions/sig
- `_exponent._bronze_epic_clarity_*.dbo_CLARITY_MEDICATION` - Medication master reference

## OMOP Fields Populated
- drug_exposure_id (surrogate key)
- drug_source_value
- drug_exposure_start_date
- sig (patient instructions)
- visit_occurrence_id (if encounter available)

In [ ]:
source = 'epic_clarity'

In [ ]:
-- Silver Layer: Transform EPIC medication order data
%sql
CREATE OR REPLACE TEMP VIEW drug_exposure_silver AS
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'ORDER_MED', 'ORDER_MED_ID', om.ORDER_MED_ID) AS drug_source_value,
    COALESCE(om.MEDICATION_ID_MEDICATION_NAME, cm.GENERIC_NAME) AS drug_name,
    om.ORDERING_DATE AS drug_exposure_start_date,
    COALESCE(oms.SIG_TEXT, '') AS sig,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC', 'PAT_ENC_CSN_ID', om.PAT_ENC_CSN_ID) AS visit_occurrence_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_ORDER_MED om
LEFT JOIN _exponent._bronze_epic_clarity_prod01_vw.dbo_ORDER_MED_SIG oms
    ON om.ORDER_MED_ID = oms.ORDER_ID
LEFT JOIN _exponent._bronze_epic_clarity_prod01_vw.dbo_CLARITY_MEDICATION cm
    ON om.MEDICATION_ID = cm.MEDICATION_ID
WHERE om.ORDER_MED_ID IS NOT NULL

In [ ]:
-- Merge into Silver Layer
%sql
MERGE INTO _exponent.omop_silver.drug_exposure AS target
USING drug_exposure_silver AS source
ON target.drug_source_value = source.drug_source_value

WHEN MATCHED AND NOT (
    target.drug_name <=> source.drug_name
    AND target.drug_exposure_start_date <=> source.drug_exposure_start_date
    AND target.sig <=> source.sig
    AND target.visit_occurrence_source_value <=> source.visit_occurrence_source_value
)
THEN UPDATE SET
    target.drug_name = source.drug_name,
    target.drug_exposure_start_date = source.drug_exposure_start_date,
    target.sig = source.sig,
    target.visit_occurrence_source_value = source.visit_occurrence_source_value,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    drug_source_value,
    drug_name,
    drug_exposure_start_date,
    sig,
    visit_occurrence_source_value,
    updated_tsp
)
VALUES (
    source.drug_source_value,
    source.drug_name,
    source.drug_exposure_start_date,
    source.sig,
    source.visit_occurrence_source_value,
    source.updated_tsp
)

In [ ]:
-- Populate mapping table
%sql
INSERT INTO _exponent.omop_mapping.source_to_drug_exposure (
    source_system,
    drug_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    'epic_clarity' AS source_system,
    s.drug_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    s.updated_tsp AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT drug_source_value, updated_tsp
    FROM _exponent.omop_silver.drug_exposure
    WHERE drug_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_drug_exposure x
    ON s.drug_source_value = x.drug_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
-- Gold Layer: Join with mapping and visit table to get drug_exposure_id and visit_occurrence_id
%sql
CREATE OR REPLACE TEMP VIEW drug_exposure_gold AS
SELECT
    m.drug_exposure_id,
    s.drug_name,
    s.drug_exposure_start_date,
    s.sig,
    mv.visit_occurrence_id,
    s.updated_tsp
FROM _exponent.omop_silver.drug_exposure s
INNER JOIN _exponent.omop_mapping.source_to_drug_exposure m
    ON s.drug_source_value = m.drug_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence mv
    ON s.visit_occurrence_source_value = mv.visit_occurrence_source_value
    AND mv.source_system = 'epic_clarity'
    AND mv.active_flag = TRUE

In [ ]:
-- Merge into Gold Layer (OMOP)
%sql
MERGE INTO _exponent.omop.drug_exposure AS target
USING drug_exposure_gold AS source
ON target.drug_exposure_id = source.drug_exposure_id

WHEN MATCHED AND NOT (
    target.drug_name <=> source.drug_name
    AND target.drug_exposure_start_date <=> source.drug_exposure_start_date
    AND target.sig <=> source.sig
    AND target.visit_occurrence_id <=> source.visit_occurrence_id
)
THEN UPDATE SET
    target.drug_name = source.drug_name,
    target.drug_exposure_start_date = source.drug_exposure_start_date,
    target.sig = source.sig,
    target.visit_occurrence_id = source.visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
    drug_exposure_id,
    drug_name,
    drug_exposure_start_date,
    sig,
    visit_occurrence_id
)
VALUES (
    source.drug_exposure_id,
    source.drug_name,
    source.drug_exposure_start_date,
    source.sig,
    source.visit_occurrence_id
)